In [ ]:
import torch
from diffusers import DiffusionPipeline, StableDiffusionXLPipeline
import os
import gc
import yaml

with open(os.path.expanduser("~/prog/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

In [ ]:
MODEL_PATH = config["paths"]["base_model_dir"]
LORA_PATH = config["paths"]["lora_model_dir"]
LORA_WEIGHTS_FILE = config["weights_file"]["lora_weights"]
OUTPUT_DIR_LORA = config["paths"]["evaluation_dir"] + "/metrics/fidelity/images_lora"
OUTPUT_DIR_WLORA = config["paths"]["evaluation_dir"] + "/metrics/fidelity/images_base"

NUM_IMAGES = 10 # Number of images per prompt

INFERENCE_STEPS = 50

prompts_tok = [
    "A professional studio photograph of TOK cat, looking at the camera, neutral grey background, high detail",
    "A detailed close-up portrait of TOK cat's face, focusing on eyes and fur pattern, soft lighting",
    "A full body shot of TOK cat sitting up straight, realistic lighting, plain ground",
    "A side view profile of TOK cat, sharp focus on head, blurred background"
]

prompts = [
    "A professional studio photograph of cat, looking at the camera, neutral grey background, high detail",
    "A detailed close-up portrait of cat's face, focusing on eyes and fur pattern, soft lighting",
    "A full body shot of cat sitting up straight, realistic lighting, plain ground",
    "A side view profile of cat, sharp focus on head, blurred background"
] # Prompts without "TOK" for the model without Lora

seeds = [42, 123, 456, 789, 101112, 131415, 161718, 192021, 222324, 252627] # Fixed seeds for reproducibility

## Load Model LoRa
The images will have this name

cat_numberOfPrompt_seed.png

In [ ]:
print("Loading Lora model...")

pipeLora = DiffusionPipeline.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.float16, 
    variant="fp16", 
    use_safetensors=True
).to("cuda")  # Carica il modello base in FP16 per risparmiare memoria GPU

pipeLora.load_lora_weights(
    LORA_PATH,
    weight_name=LORA_WEIGHTS_FILE
)

for i, prompt in enumerate(prompts_tok):
    for j in range(NUM_IMAGES):
        image_num = j+1
        current_seed = seeds[j]
        generator = torch.Generator("cuda").manual_seed(current_seed)

        filename = f"cat_{i+1:02d}_{image_num:02d}_{current_seed}.png"
        filepath = os.path.join(OUTPUT_DIR_LORA, filename)

        print(f"Generating image for prompt '{prompt}' with seed {current_seed}...")

        image = pipeLora(prompt, num_inference_steps=INFERENCE_STEPS, generator=generator).images[0]
        image.save(filepath)

## Clean cache

In [7]:
gc.collect() # Clear memory before loading the base model without Lora
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


## Load Base Model

The images will have this name

cat_numberOfPrompt_seed.png

In [9]:
print("Loading base model...")

pipe = DiffusionPipeline.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")  

for i, prompt in enumerate(prompts):
    for j in range(NUM_IMAGES):
        image_num = j+1
        current_seed = seeds[j]
        generator = torch.Generator("cuda").manual_seed(current_seed)

        filename = f"cat_{i+1:02d}_{current_seed}.png"
        filepath = os.path.join(OUTPUT_DIR_WLORA, filename)

        print(f"Generating image for prompt '{prompt}' with seed {current_seed}...")

        image = pipe(prompt, num_inference_steps=INFERENCE_STEPS, generator=generator).images[0]
        image.save(filepath)

Loading base model...


Loading pipeline components...: 100%|████████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  6.49it/s]


Generating image for prompt 'A professional studio photograph of TOK cat, looking at the camera, neutral grey background, high detail' with seed 42...


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  8.69it/s]
/homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


Generating image for prompt 'A professional studio photograph of TOK cat, looking at the camera, neutral grey background, high detail' with seed 123...


...

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  8.50it/s]


Generating image for prompt 'A side view profile of TOK cat, sharp focus on head, blurred background' with seed 222324...


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  8.51it/s]


Generating image for prompt 'A side view profile of TOK cat, sharp focus on head, blurred background' with seed 252627...


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  8.52it/s]
